# W9 Practice 2 — Retrieval-Augmented Generation over the EU AI Act

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week09/W9_practice_rag.ipynb)

**Goal.** Build a program that answers questions about the EU Artificial Intelligence Act from the text of the Act. The Act is a real regulation of about 51,000 words: 113 articles and 13 annexes. A model has seen parts of it during training, but it can mix up articles, numbers, and dates. Retrieval-augmented generation (RAG) finds the parts of the Act that are relevant to a question and gives them to the model together with the question. Chapter 8 explains the ideas; this notebook runs them on a real document.

The program has two stages. First, it prepares the document: it splits the text into **chunks** of at most 1,500 characters, turns each chunk into an **embedding** (a list of numbers, Part 3), and stores the embeddings. Then, for each question, it retrieves the chunks that are most similar to the question and asks the model to answer from them. The preparation uses three LangChain parts: a text splitter, a vector store, and a **retriever**. The answer uses the message objects and `model.invoke` of the LangChain core patterns (Week 10, Practice 1).

| Part | Question |
|---|---|
| 1. The document and a model without it | What does the model answer from memory? |
| 2. Splitting | How is a long document divided into chunks? |
| 3. Embeddings and a vector store | How are the chunks for a question found? |
| 4. Retrieval | Which chunks does a question get? |
| 5. Answering from the passages | How does a question become a grounded answer with sources? |
| 6. A question that the Act does not answer | What does the model do when no passage has the answer? |
| 7. Checking retrieval with code | How often does the search find the right article? |

**Cell labels.** The first line of each code cell says what to do with it:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.

It takes about 80 minutes and is not collected. In Week 10, the search that you build here becomes a tool of an agent, and the agent decides when to search and what to search for. The text of the Act comes from EUR-Lex, without the recitals and the footnotes; only the version in the Official Journal of the EU is legally binding.

## 0. Setup

*Do:* run the three cells; paste your API key into the second. The third cell makes the model object, as in Practice 1, and downloads the text of the Act into the file `eu_ai_act.md`.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14" "langchain-text-splitters==1.1.3"

In [ ]:
# ▶ RUN · setup: just run
import os

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

In [ ]:
# ▶ RUN · setup: just run
import urllib.request
from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1"
EMBEDDING_MODEL = "text-embedding-3-small"
model = init_chat_model(MODEL)     # one model object for the whole notebook

ACT_PATH = "eu_ai_act.md"
if not os.path.exists(ACT_PATH):   # download the text once
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ralbu85/stml_2026/main/labs/data/eu_ai_act.md", ACT_PATH)
act_text = open(ACT_PATH, encoding="utf-8").read()
print(len(act_text.split()), "words")

## 1. The document and a model without it

The file is Markdown. A line that starts with `# ` is a chapter heading, and a line that starts with `## ` is the heading of an article or an annex. Each paragraph of the Act is one line.

*Do:* run the two cells. The first prints the beginning of Article 50, to show the format: a heading line, then one numbered paragraph per line. Part 3 uses this article again. The second cell asks the model a question about the Act, with no text of the Act. Keep its answer: Part 5 asks the same question with retrieval.

In [ ]:
# ▶ RUN · example: read each line, then run
start = act_text.index("## Article 50:")          # the position where Article 50 starts
print(act_text[start:start + 700])

In [ ]:
# ▶ RUN · example: read each line, then run
QUESTION = "Under the EU AI Act, what is the maximum fine for supplying incorrect or misleading information to national authorities? Name the article."

answer_without = model.invoke(QUESTION).content    # the model alone: no documents
print(answer_without)

**Reading the output.** The answer sounds certain, and it names an article. Nothing in this call tells us whether the amounts are right: the model answered from what it learned in training. Part 5 checks the same answer against the text of the Act.

## 2. Splitting

A chunk for retrieval must be small enough to be about one thing, and it should not cut across the structure of the document. The split has two steps.

1. `MarkdownHeaderTextSplitter` cuts the text at the headings. `headers_to_split_on` gives each heading mark and the name of its label: `#` is labeled `"chapter"`, and `##` is labeled `"section"`. The splitter removes the heading lines from the text and keeps them in the **metadata** of each piece: a dictionary of labels such as `{"chapter": "Chapter IV: …", "section": "Article 50: …"}`. In this notebook, one article or one annex is called a **section**. (The Act itself also uses "Section" for groups of articles in Chapter III; this file has no headings for them.) The first piece, the title and the source note of the file, has no section label, so the code leaves it out.
2. `RecursiveCharacterTextSplitter` cuts each section that is longer than 1,500 characters into **chunks** of at most 1,500 characters. It cuts at the end of a line where it can. In this file each paragraph is one line, so no paragraph is cut. Neighboring chunks can share up to 150 characters, the **overlap**: when the last paragraph of a chunk is short, it is repeated at the start of the next chunk, so that the next chunk does not begin without context.

Each piece and each chunk is a LangChain `Document`: an object with two attributes, `page_content` (the text) and `metadata` (the labels). `split_text` takes a string; `split_documents` takes Documents and copies the metadata of a section to each of its chunks.

*Do:* run the cell. Compare the number of sections with the number of chunks, and read the metadata and the text of one chunk.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "chapter"), ("##", "section")])
pieces = header_splitter.split_text(act_text)      # one Document for each piece between two headings
sections = []
for piece in pieces:
    if "section" in piece.metadata:                # leave out the title and the source note
        sections.append(piece)
print(len(sections), "sections")

size_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
chunks = size_splitter.split_documents(sections)   # long sections become several chunks
print(len(chunks), "chunks")
print(chunks[100].metadata)
print(chunks[100].page_content)

**Reading the output.** The 126 sections are the 113 articles and the 13 annexes. 75 of them are longer than 1,500 characters, so the split makes 298 chunks. The chunk printed here begins in the middle of Article 36, with "(a) …": nothing in its text says which article it comes from. Its metadata does, because each chunk keeps the labels of its section. Part 5 uses the section label to name the source of each fact.

## 3. Embeddings and a vector store

An **embedding model** turns a text into an **embedding**: a list of numbers, also called a **vector**, that represents the meaning of the text. This notebook uses OpenAI's `text-embedding-3-small` (`EMBEDDING_MODEL`, set in Part 0), a model separate from the chat model. Texts with similar meanings get vectors that point in similar directions, and the **cosine similarity** of two vectors measures this (Chapter 8, 2.5): higher means more similar.

### 3.1 One embedding

*Do:* run the cell. It embeds a question and prints how many numbers the embedding has, and the first five of them.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)

CHATBOT_QUESTION = "Must a chatbot tell people that it is an AI?"
question_vector = embedding_model.embed_query(CHATBOT_QUESTION)   # embed_query(text): the embedding of one text
print(len(question_vector), "numbers")
print(question_vector[:5])

**Reading the output.** Every embedding of this model has 1,536 numbers, whatever the length of the text. One number alone means nothing that we can read; only the comparison of two embeddings is useful.

### 3.2 The vector store

A **vector store** embeds every chunk of the Act and keeps each embedding with its text and metadata. It gets the embedding model, so that it can embed the chunks and, later, each question. For a question, it embeds the question, computes the cosine similarity with every chunk, and returns the most similar chunks. `InMemoryVectorStore` keeps everything in the memory of this notebook, so a restart of the runtime loses it. A larger application uses a database for this, with the same calls.

*Do:* run the cell. It embeds every chunk, which takes a few seconds. Then it ranks all chunks for the question of 3.1 and prints the three most similar and the two least similar chunks, with their scores.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding_model)
ids = vector_store.add_documents(chunks)           # embeds every chunk once; returns one id for each chunk
print(len(ids), "chunks in the store")

ranking = vector_store.similarity_search_with_score(CHATBOT_QUESTION, k=len(ids))   # all chunks, most similar first
for doc, score in ranking[:3] + ranking[-2:]:      # the first three and the last two
    print(round(score, 3), doc.metadata["section"])

**Reading the output.** In our runs, the three most similar chunks all came from Article 50 (0.52, 0.47, 0.47), and the least similar ones scored almost 0 (0.01 and 0.007). The question shares almost no words with Article 50: it says "chatbot" and "tell people", and the article says "AI systems intended to interact directly with natural persons" and "informed". The embeddings still put them close. A long article has several chunks, so one article can take several of the top places. Similarity ranks the chunks; it does not prove that a chunk contains the answer.

## 4. Retrieval

**Retrieval** returns the *k* chunks that are most similar to a question. The number *k* is a choice: a small *k* gives the model less text to read, and a larger *k* gives a better chance that the answer is among the chunks.

In LangChain, a **retriever** does this. `vector_store.as_retriever(search_kwargs={"k": 3})` makes a retriever that returns the three most similar chunks, and `retriever.invoke(question)` runs it. A retriever has the same `invoke` as a model, and other parts of LangChain accept it as it is: in Week 10, one line turns it into a tool for an agent.

### 4.1 The retriever

*Do:* run the cell. `search_kwargs={"k": 3}` makes the retriever return the three most similar chunks; without it, a retriever returns 4. The cell retrieves the chunks for the question of Part 1 and prints their sections.

In [ ]:
# ▶ RUN · example: read each line, then run
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

docs = retriever.invoke(QUESTION)
for doc in docs:
    print(doc.metadata["section"])

## 5. Answering from the passages

This part builds the answering stage in four steps. Each step uses what the step before it made.

| Step | The code | It uses |
|---|---|---|
| 5.1 | `format_passages`: the retrieved chunks as one text, each under its section label | the chunks of 4.1 |
| 5.2 | the grounding rules in the system message | the passages of 5.1 |
| 5.3 | `rag_answer`: retrieve, put the passages into the input, answer | 4.1, 5.1, 5.2 |
| 5.4 | four questions and their answers | `rag_answer` |

In the input, the retrieved chunks are called **passages**.

### 5.1 The passages in the input

The splitter removed the heading lines from the text (Part 2), so the text of a chunk does not say which article it comes from. `format_passages` writes each chunk under its section label, so that the model can name its sources. LangChain leaves this step to you; its own tutorials write such a function, often called `format_docs`.

*Do:* run the cell. For each chunk, the loop adds one string: the section label in square brackets, a line break (`"\n"`), and the text of the chunk. The cell prints the first 1,200 characters of the result for the chunks of 4.1.

In [ ]:
# ▶ RUN · example: read each line, then run
def format_passages(docs):
    """Write each chunk under its section label, for example "[Article 99: Penalties]"."""
    parts = []
    for doc in docs:

        parts.append("[" + doc.metadata["section"] + "]\n" + doc.page_content)
    return "\n\n".join(parts)                     # the passages, separated by a blank line

passages = format_passages(docs)
print(passages[:1200])                             # the first 1,200 characters

### 5.2 The grounding rules in the system message

The model still has what it learned in training, and it can answer from that instead of from the passages. Chapter 8 (3.3) calls an answer **grounded** when the passages support every claim in it. The system message asks for a grounded answer with three rules:

| Rule | What it asks | Why |
|---|---|---|
| 1 | Answer only from the passages. | The passages, not the model's memory, decide the answer. |
| 2 | Name the article or annex of each fact. | A reader can check each fact in the Act. |
| 3 | If the passages do not contain the answer, say so. | The model has an honest alternative to a guess. |

The rules go into the **system message**, so they hold for every question. The passages and the question go into the **user message**. `build_messages` makes these two messages with the message objects of Practice 1 (Part 2): `SystemMessage` and `HumanMessage`.

*Do:* read the three rules in `RAG_SYSTEM` and the function `build_messages`, then run the cell. It prints the two messages for the question of Part 1: the system message with the rules and the user message with the passages. LangChain calls the user message `human`.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain_core.messages import SystemMessage, HumanMessage

# The three grounding rules, one line each (Chapter 8, 3.3)

RAG_SYSTEM = """You answer questions about the EU AI Act.
1. Answer only from the passages in the user message. Do not add facts from your own knowledge.
2. After each fact, name its article or annex, for example (Article 99(5)).
3. If the passages do not contain the answer, say so and name what is missing."""

def build_messages(passages, question):
    """The system message holds the rules; the user message holds the passages and the question."""
    system = SystemMessage(RAG_SYSTEM)                                        # the same for every question
    user = HumanMessage("Passages:\n" + passages + "\n\nQuestion: " + question)   # new for each question
    return [system, user]

messages = build_messages(format_passages(docs), QUESTION)
for message in messages:
    print("=====", message.type.upper())           # SYSTEM, then HUMAN (the user message)
    print(message.content[:1500])

### 5.3 The answer function

`rag_answer` answers one question in three steps, one line each. Each line uses a part that you already built:

1. **Retrieve:** `retriever_k.invoke(question)` returns the *k* chunks for the question (as in 4.1).
2. **Put the passages into the input:** `build_messages(format_passages(docs), question)` makes the two messages (5.1, 5.2).
3. **Answer:** `model.invoke(messages)` sends the messages to the model, as in Practice 1.

The function returns three values: the answer, the messages that the model received, and the retrieved chunks.

*Do:* read the three lines of `rag_answer`, then run the cell. It asks the question of Part 1, prints the user message and the answer, and then the answer of Part 1 without retrieval.

In [ ]:
# ▶ RUN · example: read each line, then run
def rag_answer(question, k=3):
    """Retrieve k chunks for the question, put them into the input, and answer from them."""
    retriever_k = vector_store.as_retriever(search_kwargs={"k": k})

    docs = retriever_k.invoke(question)                            # 1. retrieve
    messages = build_messages(format_passages(docs), question)     # 2. put the passages into the input
    reply = model.invoke(messages)                                 # 3. answer
    return reply.content, messages, docs

answer_k3, messages_k3, docs_k3 = rag_answer(QUESTION)
print(messages_k3[1].content)                      # the user message: the passages and the question
print("=" * 80)
print("ANSWER WITH RETRIEVAL:", answer_k3)
print()
print("ANSWER WITHOUT RETRIEVAL:", answer_without)

**Reading the output.** In our runs, the three chunks were a middle chunk of Article 100 (fines on EU institutions), the first chunk of Article 101 (fines for providers of general-purpose AI models), and the first chunk of Article 99, with paragraphs 1 to 3. All three are about fines, which is why they scored highest, but none of them has the fine for incorrect information to national authorities. That fine is in paragraph 5, in the next chunk of Article 99. The input did not contain the answer, and the model said so instead of filling the gap from memory, as rule 3 asks. Chapter 8 calls this a **retrieval error**: the text that the answer needs never reached the model. 5.5 shows how to get that chunk into the input.

In our runs, the answer without retrieval was certain and wrong. In all three runs it gave EUR 10 000 000 or 2 %, named a different paragraph each time (Article 99(2)(b), 99(3), and 99(5)), and quoted sentences that are not in the Act. The Act says EUR 7 500 000 or 1 %, in Article 99(5). A model without the text can name the right article and still give the amounts of an earlier draft, and it sounds just as certain.

### 5.4 Questions and answers

Now use `rag_answer` as an assistant. The cell asks three questions and one question of your own. For each question it prints the sections of the passages and the answer, so you can check the three rules:

- Does the answer name an article or annex (rule 2)?
- Is that article or annex among the printed sections, that is, did the answer come from the passages (rule 1)?
- If the sections do not hold the answer, does the answer say so (rule 3)?

*Do:* run the cell and check the three rules for each answer. Then change `MY_QUESTION` to a question of your own, for example about emotion recognition or the duties of deployers, and run the cell again.

In [ ]:
# ▶ RUN · example: read each line, then run
MY_QUESTION = "Must deployers tell people when an image or video is a deepfake?"   # change this to your own question

QUESTIONS = [
    CHATBOT_QUESTION,                                                   # from Part 3
    "Which AI practices are prohibited?",
    "What documentation must providers of general-purpose AI models keep up to date?",
    MY_QUESTION,
]
for question in QUESTIONS:
    answer, messages, docs = rag_answer(question)
    print("QUESTION:", question)
    print("SECTIONS:")
    for doc in docs:
        print("   ", doc.metadata["section"])
    print("ANSWER:  ", answer)
    print("-" * 80)

### 5.5 One more chunk

5.3 found no answer with `k=3`. The chunk with paragraph 5 of Article 99 was close, but not among the three.

*Do:* run the cell. It prints the five chunks that are most similar to the question of Part 1, with their scores, and marks the chunk that holds paragraph 5. Then it asks again with `k=5`.

In [ ]:
# ▶ RUN · example: read each line, then run
def show_ranking(query, k):
    """Print the k most similar chunks with their scores; mark the chunk with paragraph 5 of Article 99."""
    for doc, score in vector_store.similarity_search_with_score(query, k=k):
        mark = ""
        if "5. The supply of incorrect" in doc.page_content:
            mark = "<- paragraph 5"
        print(round(score, 3), doc.metadata["section"], mark)

show_ranking(QUESTION, 5)
answer_k5, messages_k5, docs_k5 = rag_answer(QUESTION, k=5)
print()
print("ANSWER WITH k=5:", answer_k5)

**Reading the output.** In our runs, the chunk with paragraph 5 was fourth, and its score was almost the same as the score of the third chunk (0.636 against 0.639). With `k=5` it is in the input, and the answer gives EUR 7 500 000 or 1 % and names Article 99(5). A cut-off at *k* can drop a chunk that is almost as similar as the last chunk kept. A larger *k* also makes the input longer: in our runs, 895 input tokens with `k=3` and 1,466 with `k=5`, against about 62,000 for the whole Act.

The question itself was part of the problem. Its embedding represents the whole question, including "Under the EU AI Act" and "Name the article", not only the fine for incorrect information. A short search query that keeps only the topic can rank the right chunk higher.

*Try:* run the cell below. It ranks the chunks for the short query "fine for supplying incorrect information".

In [ ]:
# ▶ RUN · example: read each line, then run
show_ranking("fine for supplying incorrect information", 3)

**Reading the output.** In our runs, the chunk with paragraph 5 came first for the short query. The question that a user asks and the search query that works best are not always the same text. In Week 10, an agent writes its own search query.

## 6. A question that the Act does not answer

The search always returns *k* chunks, even when none of them answers the question. The system message tells the model what to do then.

*Do:* run the cell. The Act uses the term "SMEs" (small and medium-sized enterprises), but its articles do not say how many employees an SME may have. Read the passages and the answer. Does the answer say that the passages do not give the number, or does it give a number?

In [ ]:
# ▶ RUN · example: read each line, then run
SME_QUESTION = "How many employees can a company have and still count as an SME under the EU AI Act?"

sme_answer, sme_messages, sme_docs = rag_answer(SME_QUESTION)
print(sme_messages[1].content)                     # the passages and the question that the model received
print("=" * 80)
print(sme_answer)

**Reading the output.** In our runs, the passages came from Articles 70, 62, and 63, and the model answered that they do not give the number. Article 63 even names the source of the definition, "Recommendation 2003/361/EC", but the numbers are in that Recommendation, not in the Act. "The passages do not say" is the correct answer here. If your answer gives a number, check whether it is in the passages. A number that is not in the passages comes from what the model learned in training, even if it is correct. For an assistant about a regulation, an answer must come from the text that it cites.

## 7. Checking retrieval with code

A RAG answer can only be as good as the chunks that the search finds. Chapter 8 measured retrieval with **recall@k**: the share of the needed chunks that appear among the top *k*. Here each question needs one section, so each question is a **hit** (its section is among the *k* chunks found) or a **miss**. The **hit rate at k** (hit@k) is the share of questions with a hit. It is the code grader of Week 5, applied to retrieval.

`EVAL_SET` holds questions with the article or annex that answers each one, found by reading the Act. The check has one limit: it compares sections, not chunks. For the question of Part 5, Article 99 was among the three chunks (4.1), so hit@3 counts it as a hit, although the chunk with paragraph 5 was missing. A hit is needed for a correct answer, but it does not guarantee one.

*Do:* run the cell. It defines `hit_rate`, which prints one line for each question and then the hit rate, and runs it with `k=3`.

In [ ]:
# ▶ RUN · example: read each line, then run
EVAL_SET = [
    ("May an employer use AI to infer the emotions of its employees?", "Article 5"),
    ("Is an AI system that screens job applications a high-risk AI system?", "Annex III"),
    ("Must a chatbot inform people that they are interacting with an AI system?", "Article 50"),
    ("What documentation must providers of general-purpose AI models keep up to date?", "Article 53"),
    ("What is the maximum fine for using a prohibited AI practice?", "Article 99"),
    ("From which date do the prohibitions of AI practices apply?", "Article 113"),
]

def hit_rate(k):
    """Print, for each question in EVAL_SET, whether its section is among the k chunks found."""
    hits = 0
    for question, expected in EVAL_SET:
        found = []
        for doc in vector_store.as_retriever(search_kwargs={"k": k}).invoke(question):
            found.append(doc.metadata["section"].split(":")[0])   # "Article 5: Prohibited ..." -> "Article 5"
        if expected in found:
            hits = hits + 1
            print("hit  | expected:", expected, "| found:", found)
        else:
            print("MISS | expected:", expected, "| found:", found)
    print("hit@" + str(k) + ":", hits, "/", len(EVAL_SET))

hit_rate(3)

**Reading the output.** In our runs, five of the six questions were hits. The miss shows a limit of similarity search: the passage that answers a question can use different words from the question. Article 113 gives the dates as "Chapters I and II shall apply from 2 February 2025". It names the prohibitions only by the number of their chapter, so nothing in its text carries the meaning "prohibition".

### 7.1 One more question

*Do:* run the cell. It adds a question about scientific research, with the section that answers it (Article 2), and runs `hit_rate(3)` again. To add a question of your own, find the answer in the text first (print an article as in Part 1, or open `eu_ai_act.md` in the Files panel of Colab), and write the label as it appears before the colon of a section label, for example `"Article 6"` or `"Annex III"`.

In [ ]:
# ▶ RUN · example: read each line, then run
EVAL_SET.append(("Does the AI Act apply to AI systems used only for scientific research?", "Article 2"))

hit_rate(3)

### 7.2 Another k

*Do:* run the cell with `K = 1`, then change it to `5` and run it again. Which misses become hits? For the cost of a larger *k*, see 5.5.

In [ ]:
# ▶ RUN · example: read each line, then run
K = 1                                              # change this value and run the cell again
hit_rate(K)

**Reading the output.** In our runs, with the seven questions of 7.1, hit@1 was 3 of 7 and hit@5 was 6 of 7, the same as hit@3. At `k=1`, the documentation question got Annex XI first. Annex XI is the documentation that Article 53(1)(a) refers to, so it also answers the question, but the evaluation set allows only one section and counts it as a miss. The question about Article 113 stayed a miss at every *k* that we tried: a larger *k* does not help when the right chunk does not look similar to the question. In Week 11, the program rewrites such a question and searches again.